# Shared stream + observers

Yesterday's pipeline ran — but we propagated context by hand, stuffing each worker's summary into the next worker's prompt. That throws away everything except the summary: the analyst never sees what the researcher actually *searched for*, and any citation in the final report is downstream of our serialisation.

Two changes today fix both problems:

1. **One `MemoryStream` shared across all workers.** Every event — model messages, tool calls, tool results — lands on the same stream, so later workers read the real conversation instead of our hand-rolled summary.
2. **An `@observer` decorator** that runs every time a worker emits a `ToolResultsEvent`. We use it to record every `title → url` pair the search returned, then ground every cited title against that dict at render time.

In [1]:
import os
from typing import Literal

from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from autogen.beta import Agent, MemoryStream
from autogen.beta.config import OpenAIConfig
from autogen.beta.events import ToolResultsEvent
from autogen.beta.tools import ExaToolkit

load_dotenv()

config = OpenAIConfig(
    model="gpt-5.4-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.openai.com/v1",
)
exa = ExaToolkit(api_key=os.getenv("EXA_API_KEY"))

## Schemas (same as Day 3)

In [2]:
class Citation(BaseModel):
    title: str = Field(description="Exact title of a search result you're citing — must match a hit returned by your tools.")
    url: str = Field(description="URL of that source.")
    relevance: str = Field(description="One sentence on why this source supports your claim.")

class Finding(BaseModel):
    topic: str = Field(description="The specific aspect this finding addresses.")
    summary: str = Field(description="2-3 sentences capturing the core finding.")
    citations: list[Citation] = Field(description="Cite only sources that came back from your search tool — never invent. Emit [] if you didn't search.")
    novelty: float = Field(ge=0.0, le=1.0, description="0 = textbook, 1 = cutting edge.")

WorkerRole = Literal["researcher", "analyst", "critic"]

class WorkerSpec(BaseModel):
    role: WorkerRole
    name: str = Field(description="Short lowercase identifier, no spaces.")
    prompt: str = Field(description="System prompt for this worker, focused on its role.")
    tools: list[str] = Field(description="Tool keys. Use 'exa_search' for researchers; emit [] for none.")

class ResearchPlan(BaseModel):
    topic: str
    description: str
    team: list[WorkerSpec] = Field(description="Pipeline order: researcher → analyst → critic.")

## Plan the team

In [3]:
planner = Agent(
    name="planner",
    prompt=(
        "You design tight 3-person research pipelines: researcher → analyst → critic, in that order. "
        "The researcher gets exa_search; analyst and critic need no tools and reason over upstream findings."
    ),
    config=config,
    response_schema=ResearchPlan,
)

plan: ResearchPlan = await (await planner.ask(
    "Plan a small team to investigate high-Tc superconductivity."
)).content()

## Shared stream + observer

All three workers share `shared_stream`. The researcher's tool calls, tool results, and model replies all land there, and the analyst and critic read off the same stream — no prompt-stuffing.

`@worker.observer(ToolResultsEvent)` registers a sync callback that fires every time the worker emits a tool-results event. We duck-type our way into the search hits (any `.results` field on the data, any `.title` and `.url` on each hit) so the same observer works against any search-style toolkit, not just Exa.

In [4]:
TOOLBOX = {"exa_search": [exa.search()]}

shared_stream = MemoryStream()
title_to_url: dict[str, str] = {}

def norm(s: str) -> str:
    return s.lower().strip()

async def run_worker(spec: WorkerSpec, topic: str) -> Finding | str:
    typed = spec.role == "researcher"
    schema_note = (
        "Every Citation.title must be the exact title of a real search result that appears in this conversation — never invent sources."
        if typed else
        "Read what the prior workers said in the conversation and write a concise markdown response."
    )
    tools = [t for key in spec.tools for t in TOOLBOX.get(key, [])]
    worker = Agent(
        name=spec.name,
        prompt=spec.prompt,
        config=config,
        tools=tools,
        response_schema=Finding if typed else None,
    )

    @worker.observer(ToolResultsEvent)
    def _collect(event: ToolResultsEvent) -> None:
        for r in event.results:
            data = r.result.parts[0].data
            for hit in getattr(data, "results", None) or []:
                title, url = getattr(hit, "title", None), getattr(hit, "url", None)
                if title and url:
                    title_to_url[title] = url

    reply = await worker.ask(
        f"Topic: {topic}\n\nContribute as {spec.role}. {schema_note}",
        stream=shared_stream,
    )
    return await reply.content() if typed else (reply.body or "")

results: list[Finding | str] = []
for spec in plan.team:
    results.append(await run_worker(spec, plan.topic))

## Render — every link comes from the source

Match each `Citation.title` against the dict the observer filled, case- and whitespace-insensitive. A match (✓) renders a real hyperlink built from the URL the search actually returned; a miss (✗) shows the title alone. The agent's own `Citation.url` is never trusted at render time — only the observer-captured one.

In [5]:
norm_to_real = {norm(t): (t, u) for t, u in title_to_url.items()}

lines: list[str] = []
for spec, result in zip(plan.team, results):
    lines.append(f"### {spec.name} ({spec.role})")
    if isinstance(result, Finding):
        lines.append(f"*{result.topic} — novelty={result.novelty:.2f}*")
        lines.append(result.summary)
        for c in result.citations:
            match = norm_to_real.get(norm(c.title))
            if match:
                real_title, url = match
                lines.append(f"- ✓ [{real_title}]({url}) — {c.relevance}")
            else:
                lines.append(f"- ✗ {c.title} — {c.relevance}")
    else:
        lines.append(result)
    lines.append("")
display(Markdown("\n".join(lines)))

### scout (researcher)
*Cuprate superconductors: leading signatures and open mechanism questions — novelty=1.00*
Recent reviews and primary reports agree that cuprates remain the canonical unconventional high-Tc family: superconductivity lives in the CuO2 planes, the gap symmetry is overwhelmingly consistent with d-wave, and charge order/charge-density-wave correlations are ubiquitous. There is also growing experimental support that the pseudogap is closely tied to pairing in at least some cuprates, but whether it is a precursor pairing scale, a competing-order gap, or both in different regimes remains unresolved.
- ✗ Charge Correlations in Cuprate Superconductors — 2024 review summarizing charge-density-wave order as a near-universal property of cuprates, including its probes, symmetry, and debated connection to superconductivity.
- ✗ High-temperature superconductivity — Nature Reviews Physics consensus-style article stating that no microscopic theory is yet established and highlighting the central role of strong correlations and open questions across cuprates.
- ✗ Feshbach hypothesis of high-Tc superconductivity in cuprates — 2025 theoretical paper summarizing the broad agreement that magnetism is central, while also noting the established d-wave symmetry and the debated pseudogap mechanism.
- ✗ Spectroscopic Signature of Electronic Pairing in the Normal State of Cuprate Superconductors — 2024 report describing a normal-state gap in Nd2-xCexCuO4 that is interpreted as pairing-related and therefore relevant to the pseudogap/pairing debate.
- ✗ The Physics of Pair-Density Waves: Cuprate Superconductors and Beyond — Review of PDW order in cuprates, emphasizing that evidence exists but remains incomplete and that its relation to charge order and superconductivity is still debated.

### synth (analyst)
## High-Tc superconductivity: concise landscape

### 1) Material families and what is established
- **Cuprates** remain the best-studied ambient-pressure high-Tc family, with superconductivity confined to the **CuO2 planes** and a highly robust **d-wave** pairing signature.
- **Iron-based superconductors** form a separate unconventional family with multiple orbital characters and a more diverse gap landscape, but a common theme of **spin-fluctuation-driven pairing**.
- **Hydrides under pressure** are the clearest case of **conventional, phonon-mediated** high-Tc superconductivity reaching record temperatures, but only at **megabar pressures**.

### 2) Mechanisms: strongest evidence vs hypotheses
#### Cuprates
- **Strong evidence**
  - d-wave pairing symmetry.
  - Charge order / CDW is widespread and likely intrinsic to the phase diagram.
  - Strong electronic correlations are central.
- **Mixed / debated**
  - Whether the pseudogap is a **preformed-pair state**, a **competing order** state, or a mix depending on doping/material.
  - Whether charge order is a primary driver or a consequence of the correlated electronic background.
- **Emerging hypothesis**
  - Recent spectroscopy suggests the pseudogap may be directly tied to **pair formation**, with **phase coherence** limiting Tc in at least some cuprates.

#### Iron-based superconductors
- **Strong evidence**
  - Unconventional superconductivity with multiple gaps and strong orbital dependence.
  - In many families, data are consistent with **s± pairing** and a central role for **spin fluctuations**.
- **Mixed / debated**
  - Exact gap symmetry varies by composition, especially in heavily hole-doped or hole-pocket-only systems.
  - The balance between spin fluctuations, orbital selectivity, nematicity, and correlations is not fully settled.
- **Pattern**
  - Compared with cuprates, Fe-based systems are less universally dominated by a single motif, but they still show a recurring link between magnetism and pairing.

#### Hydrides
- **Strong evidence**
  - High Tc is reproducible in several compressed hydride systems.
  - The isotope effect, pressure dependence, and magnetic-field response support **electron-phonon pairing**.
- **Mixed / missing**
  - The main challenge is not pairing theory but **phase stability, synthesis, and characterization under extreme pressure**.
  - Ambient-pressure pathways remain largely hypothetical.

### 3) Cross-family patterns
- High Tc tends to emerge near **instabilities**:
  - cuprates: Mott/charge/spin competition
  - iron-based: magnetic/nematic/orbital competition
  - hydrides: structural and lattice-instability engineering under pressure
- **Low-dimensionality** and **strong coupling** recur across cuprates and Fe-based systems.
- **Competing or intertwined orders** are common in unconventional families, whereas hydrides are more consistent with a cleaner phonon-mediated picture.

### 4) Practical implications
- **For experiments**
  - Cuprates: prioritize probes that separate **pairing scale** from **phase coherence** and resolve the role of charge order/pseudogap.
  - Fe-based systems: map pairing symmetry with momentum resolution across compositions, especially in orbitally selective regimes.
  - Hydrides: focus on **reproducible characterization under pressure** and routes to **pressure reduction**.
- **For theory**
  - Cuprates still demand a unified theory that can incorporate **Mott physics, pairing, and intertwined orders**.
  - Fe-based superconductors need models that reconcile **multiorbital electronic structure** with gap-symmetry diversity.
  - Hydrides are a benchmark for predictive **ab initio superconductivity**, but require better modeling of **stability and anharmonicity**.

### 5) Bottom line
- **Cuprates and iron-based superconductors are still fundamentally unresolved unconventional systems.**
- **Hydrides are better understood mechanistically, but far less practical due to pressure.**
- The most promising route forward is to treat high-Tc superconductivity as a **family of related but not identical problems**, rather than expecting one universal mechanism to fit all materials.

### skeptic (critic)
## Critique of the prior synthesis

The summary is directionally reasonable, but it overstates several points as settled when the evidence is still mixed.

### Main overclaims
- **“d-wave pairing in cuprates is robust”**: pairing symmetry is indeed well supported, but this does not resolve whether the pseudogap, charge order, and superconductivity share one mechanism.
- **“spin-fluctuation-driven pairing in iron-based superconductors”**: this is a leading interpretation, not a consensus proof. The symmetry and glue vary across compositions, especially in hole-only systems.
- **“hydrides are cleaner and conventional”**: mostly true for the leading hydride cases, but the claim glosses over how much of the field still hinges on challenging high-pressure characterization and disputed data quality in some studies.

### Missing alternatives / ambiguities
- **Cuprates**
  - The pseudogap could still be a **competing-order phenomenon**, a **preformed-pair regime**, or a **hybrid** depending on doping/material.
  - Charge order may be **causal, consequential, or merely coexisting**; the summary treats it as established background.
- **Iron-based superconductors**
  - Orbital selectivity, nematicity, and local-correlation scenarios are underweighted relative to the spin-fluctuation story.
  - The “common theme” language hides real family-to-family variation in gap structure.
- **Hydrides**
  - The main bottleneck may be **materials synthesis and metastability**, not mechanism.
  - The summary underplays the impact of **reproducibility and characterization disputes** on field confidence.

### What would falsify the leading explanations?
- **Cuprates**
  - If pairing-sensitive probes showed that the pseudogap persists independently of pairing onset across multiple materials and dopings, the “preformed pairs” interpretation weakens.
  - If charge order were shown to vanish without changing Tc or the pseudogap scale, the centrality of charge order would weaken.
- **Iron-based superconductors**
  - If high-resolution gap mapping consistently showed no sign-changing structure in systems widely assumed to be s±, the spin-fluctuation narrative would be challenged.
- **Hydrides**
  - If independent groups failed to reproduce Meissner, transport, and isotope signatures under identical pressure conditions, the conventional hydride story would need revision.

### Most ambiguous data
- Cuprate pseudogap spectroscopy: pairing-like gaps can also be mimicked by other correlated states.
- Charge-order measurements: static vs dynamic order is still hard to disentangle.
- Hydride high-pressure transport and magnetic screening: extreme experimental conditions make artifacts hard to exclude.

### Key experiments that would reduce uncertainty most
1. **Cuprates:** simultaneous, spatially resolved measurements of pairing, charge order, and phase coherence on the same sample and doping range.
2. **Iron-based:** momentum-resolved gap symmetry and phase-sensitive tests across the full compositional range, especially hole-pocket-only materials.
3. **Hydrides:** fully independent replication with cross-lab pressure calibration, isotope substitution, and simultaneous transport + Meissner detection.

## Risk assessment
- **Low risk:** the broad family-level distinctions in the summary.
- **Medium risk:** the implied mechanistic hierarchy within cuprates and iron-based superconductors.
- **High risk:** any suggestion that one mechanism is likely to explain all high-Tc materials.

## Unresolved questions
- Is the cuprate pseudogap a precursor to superconductivity, a competing order, or both?
- Is charge order central to pairing or a byproduct of strong correlations?
- What is the dominant pairing glue in iron-based superconductors across all compositions?
- Can hydride superconductivity be made reproducible and practical outside megabar conditions?
- Is there any genuinely universal organizing principle behind high-Tc superconductivity, or only family-specific ones?


## Up next

Citations are now grounded — every ✓ link is one the search engine actually returned. But validation is still purely structural: Pydantic catches a missing `url` field, the observer catches a hallucinated title. What if you want to enforce *content* rules — "this paragraph must use first-person plural", "at least two citations" — and have the agent retry until it complies? Tomorrow: schema-backed retries.